In [1]:
import os
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
os.getcwd()

'C:\\Users\\TWH\\Smart-city-traffic-capstone-\\part3_machine_learning\\notebooks'

In [2]:
# 1. Load Data
df = pd.read_csv("data/processed/cleaned_traffic_features.csv")
feature_cols = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "is_weekend",
    "is_holiday",
    "is_severe_weather",
    "is_low_visibility",
    "temp_scaled",
    "clouds_scaled",
]

X = df[feature_cols]
y = df["traffic_volume"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("Traffic_Volume_Model_Lifecycle")

# Define model iterations for versioning
model_candidates = [
    (
        "v1_Ridge_Baseline",
        Ridge(alpha=1.0),
        {"alpha": 1.0, "model_type": "Ridge"},
    ),
    (
        "v2_RandomForest_Tuned",
        RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
        {"n_estimators": 100, "max_depth": 12, "model_type": "RandomForest"},
    ),
    (
        "v3_GradientBoosting_Champion",
        GradientBoostingRegressor(
            n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42
        ),
        {
            "n_estimators": 150,
            "learning_rate": 0.1,
            "max_depth": 6,
            "model_type": "GradientBoosting",
        },
    ),
]

2026/09/25 13:41:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 13:41:44 INFO mlflow.store.db.utils: Updating database tables
2026/09/25 13:41:45 INFO mlflow.tracking.fluent: Experiment with name 'Traffic_Volume_Model_Lifecycle' does not exist. Creating a new experiment.


In [3]:
# 2. Iterative Experimentation & Model Registry
for run_name, model, params in model_candidates:
    with mlflow.start_run(run_name=run_name) as run:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        # Log parameters and metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"MAE": mae, "R2_Score": r2})

        # Register model binary into MLflow Model Registry
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name="Traffic_Volume_Regressor",
        )
        print(f"Logged {run_name} | MAE: {mae:.2f} | R2: {r2:.4f}")

2026/09/25 13:41:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Traffic_Volume_Regressor'.
Created version '1' of model 'Traffic_Volume_Regressor'.


Logged v1_Ridge_Baseline | MAE: 835.00 | R2: 0.7086


2026/09/25 13:42:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'Traffic_Volume_Regressor' already exists. Creating a new version of this model...
Created version '2' of model 'Traffic_Volume_Regressor'.


Logged v2_RandomForest_Tuned | MAE: 265.65 | R2: 0.9470


2026/09/25 13:42:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged v3_GradientBoosting_Champion | MAE: 266.26 | R2: 0.9492


Registered model 'Traffic_Volume_Regressor' already exists. Creating a new version of this model...
Created version '3' of model 'Traffic_Volume_Regressor'.
